### First, install ijson to handle big json files.

In [1]:
%pip install ijson

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Please put this script at the root of the project, with at least the following json in a folder named "data" :
- song.json
- album.json
- artist-without-members.json

## Here is the script to filter songs data and group them by year

It took less than 10 minutes to run on my computer so it should be done in one go.

In [2]:
import ijson
import json
import os
from collections import defaultdict

# List of keys to keep, except releaseDate (we will keep releaseYear)
keys_to_keep = ["_id", "id_album", "publicationDate", "title", "name", "genre", "explicitLyrics", "isClassic"]  # Adjust this list based on the keys you want to keep
song_done = 0
output_folder = "data/processed_songs"  # Base output folder
batch_size = 10000  # Max number of songs per file

# Ensure the output folder exists
os.makedirs(output_folder, exist_ok=True)

# Dictionary to keep track of file handles, song counts, and file batch number for each year
file_info = defaultdict(lambda: {"file_handle": None, "song_count": 0, "batch_number": 1, "first_item": True})

def get_year_from_song(song):
    """Extract the year from the song's publication date or album date, handle '????' as empty."""
    if 'publicationDate' in song and song['publicationDate'] and not '?' in song['publicationDate'][:4] and 1000 < int(song['publicationDate'][:4]) < 2021:
        return song['publicationDate'][:4]  # Extract year (YYYY)
    elif 'publicationDateAlbum' in song and song['publicationDateAlbum'] and not '?' in song['publicationDateAlbum'][:4] and 1000 < int(song['publicationDateAlbum'][:4]) < 2021:
        return song['publicationDateAlbum'][:4]  # Assume it's just a year
    return "Unknown"

def get_output_file_for_year(year):
    """Get or create the file for a specific year, batching them to 10,000 songs max per file."""
    year_info = file_info[year]
    
    # Check if we need to start a new batch file
    if year_info["file_handle"] is None or year_info["song_count"] >= batch_size:
        if year_info["file_handle"]:
            # Close the previous file's JSON array
            year_info["file_handle"].write(']')
            year_info["file_handle"].close()

        # Prepare the new file
        batch_number = year_info["batch_number"]
        year_folder = os.path.join(output_folder, year)
        os.makedirs(year_folder, exist_ok=True)  # Ensure year folder exists
        new_file_path = os.path.join(year_folder, f"{year}_filtered_songs_{batch_number}.json")
        
        # Open the new file and write the beginning of the array
        year_info["file_handle"] = open(new_file_path, 'w', encoding='utf-8')
        year_info["file_handle"].write('[')
        
        # Reset song count and increment batch number for the next file
        year_info["song_count"] = 0
        year_info["batch_number"] += 1
        year_info["first_item"] = True  # Reset first item for the new batch
    
    return year_info["file_handle"]

def close_all_files():
    """Close all open file handles and properly end JSON arrays."""
    for year, info in file_info.items():
        if info["file_handle"]:
            info["file_handle"].write(']')  # Close the JSON array
            info["file_handle"].close()

def fill_missing_fields_and_add_year(song, year, keys_to_keep):
    """Fill missing fields with empty strings and attempt to fill genre from album_genre if needed. Add releaseYear."""
    filtered_song = {}
    for key in keys_to_keep:
        if key == "genre":
            # Handle genre: use genre if present, otherwise fall back to album_genre
            filtered_song[key] = song.get("genre", "") or song.get("album_genre", "")
        else:
            # Fill with an empty string if the key is missing
            filtered_song[key] = song.get(key, "")
    # Add releaseYear field
    filtered_song["releaseYear"] = year
    return filtered_song

with open('data/song.json', 'r', encoding='utf-8') as input_file:
    # Parse the JSON array incrementally
    parser = ijson.items(input_file, 'item')

    song_done = 0

    try:
        while True:  # We'll increment the counter inside the loop
            song = next(parser)  # Get the next song
            
            # Extract the relevant year
            year = get_year_from_song(song)

            # Filter the song data based on the keys to keep, handling missing fields, and adding releaseYear
            filtered_song = fill_missing_fields_and_add_year(song, year, keys_to_keep)

            # Get the output file for the year
            output_file = get_output_file_for_year(year)

            # Write to the appropriate file for the year
            if not file_info[year]["first_item"]:
                output_file.write(',')  # Add a comma before each new item after the first one
            else:
                file_info[year]["first_item"] = False  # Set this to False after the first item for the year

            # Write the filtered song as a JSON object
            json.dump(filtered_song, output_file, indent=4)

            # Increment the song counter
            song_done += 1
            file_info[year]["song_count"] += 1  # Increment the song count for the current file

            # Print status every 10,000 songs
            if song_done % 10000 == 0:
                print(f"{song_done} songs processed.")

    except StopIteration:
        # When we reach the end of the JSON file, the iteration stops
        pass

# Close all open files after processing
close_all_files()

print(f"Filtered JSON files written to '{output_folder}'. Total songs processed: {song_done}")


10000 songs processed.
20000 songs processed.
30000 songs processed.
40000 songs processed.
50000 songs processed.
60000 songs processed.
70000 songs processed.
80000 songs processed.
90000 songs processed.
100000 songs processed.
110000 songs processed.
120000 songs processed.
130000 songs processed.
140000 songs processed.
150000 songs processed.
160000 songs processed.
170000 songs processed.
180000 songs processed.
190000 songs processed.
200000 songs processed.
210000 songs processed.
220000 songs processed.
230000 songs processed.
240000 songs processed.
250000 songs processed.
260000 songs processed.
270000 songs processed.
280000 songs processed.
290000 songs processed.
300000 songs processed.
310000 songs processed.
320000 songs processed.
330000 songs processed.
340000 songs processed.
350000 songs processed.
360000 songs processed.
370000 songs processed.
380000 songs processed.
390000 songs processed.
400000 songs processed.
410000 songs processed.
420000 songs processed.
4

### Please load this set of countries that we'll need for filtering artists

In [3]:
# Set of known countries
countries = set([
    "afghanistan", "albania", "algeria", "american samoa", "andorra", "angola", "antigua and barbuda", 
    "argentina", "armenia", "australia", "austria", "azerbaijan", "bahamas", "bahrain", "bangladesh", 
    "barbados", "belarus", "belgium", "belize", "benin", "bhutan", "bolivia", "bosnia and herzegovina", 
    "botswana", "brazil", "brunei darussalam", "bulgaria", "burkina faso", "burundi", "cabo verde", 
    "cambodia", "cameroon", "canada", "central african republic", "chad", "chile", "china", "colombia", 
    "comoros", "congo", "costa rica", "croatia", "cuba", "cyprus", "czech republic", "czechoslovakia", 
    "democratic people's republic of korea", "democratic republic of the congo", "denmark", "djibouti", 
    "dominica", "dominican republic", "ecuador", "egypt", "el salvador", "equatorial guinea", "eritrea", 
    "estonia", "eswatini", "ethiopia", "fiji", "finland", "france", "gabon", "gambia", "georgia", "germany", 
    "ghana", "gibraltar", "greece", "greenland", "grenada", "guatemala", "guinea", "guinea-bissau", 
    "guyana", "haiti", "honduras", "hungary", "iceland", "india", "indonesia", "iran", "iraq", "ireland", 
    "israel", "italy", "ivory coast", "jamaica", "japan", "jordan", "kazakhstan", "kenya", "kiribati", 
    "kosovo", "kuwait", "kyrgyzstan", "laos", "latvia", "lebanon", "lesotho", "liberia", "libya", 
    "liechtenstein", "lithuania", "luxembourg", "madagascar", "malawi", "malaysia", "maldives", "mali", 
    "malta", "marshall islands", "mauritania", "mauritius", "mexico", "micronesia", "moldova", "monaco", 
    "mongolia", "montenegro", "morocco", "mozambique", "myanmar", "namibia", "nauru", "nepal", 
    "netherlands", "new zealand", "nicaragua", "niger", "nigeria", "north korea", "north macedonia", 
    "northern cyprus", "norway", "oman", "pakistan", "palau", "panama", "papua new guinea", "paraguay", 
    "peru", "philippines", "poland", "portugal", "puerto rico", "qatar", "romania", "russia", "rwanda", 
    "saint kitts and nevis", "saint lucia", "saint vincent and the grenadines", "samoa", "san marino", 
    "sao tome and principe", "saudi arabia", "senegal", "serbia", "seychelles", "sierra leone", 
    "singapore", "slovakia", "slovenia", "solomon islands", "somalia", "south africa", "south korea", 
    "south sudan", "spain", "sri lanka", "sudan", "suriname", "sweden", "switzerland", "syria", 
    "taiwan", "tajikistan", "tanzania", "thailand", "timor-leste", "togo", "tonga", "trinidad and tobago", 
    "tunisia", "turkey", "turkmenistan", "tuvalu", "uganda", "ukraine", "united arab emirates", 
    "united kingdom", "united states", "uruguay", "uzbekistan", "vanuatu", "vatican city", "venezuela", 
    "vietnam", "yemen", "zambia", "zimbabwe", "republic of kosovo", "cape verde", "bermuda", "anguilla",
    "[worldwide]"
])
print(len(countries))

208


## Here is the script to filter artists data and group them by countries

In [4]:
import ijson
import json
import os
from collections import defaultdict

# List of keys to keep for each artist
keys_to_keep = ["_id", "name", "recordLabel", "deezerFans"]  # Adjust this list based on the keys you want to keep
artist_done = 0
output_folder = "data/processed_artists"  # Base output folder for artists
batch_size = 1000  # Max number of artists per file

# Ensure the output folder exists
os.makedirs(output_folder, exist_ok=True)

# Dictionary to keep track of file handles, artist counts, and batch numbers for each country
file_info = defaultdict(lambda: {"file_handle": None, "artist_count": 0, "batch_number": 1, "first_item": True})

def get_country_from_artist(artist):
    """Extract the country from the artist's location or locationInfo."""
    if 'location' in artist and 'country' in artist['location'] and artist['location']['country'] and artist['location']['country'].lower() in countries:
        return artist['location']['country']  # Return the country from the location field
    elif 'locationInfo' in artist and len(artist['locationInfo']) > 0 and artist['locationInfo'][0].lower() in countries:
        return artist['locationInfo'][0]  # Use the first entry in locationInfo as the country
    return "Unknown"  # Default to 'Unknown' if no country is found

def get_output_file_for_country(country):
    """Get or create the file for a specific country, batching them to 1,000 artists max per file."""
    country_info = file_info[country]
    
    # Check if we need to start a new batch file
    if country_info["file_handle"] is None or country_info["artist_count"] >= batch_size:
        if country_info["file_handle"]:
            # Close the previous file's JSON array
            country_info["file_handle"].write(']')  # Close the JSON array
            country_info["file_handle"].close()

        # Prepare the new file
        batch_number = country_info["batch_number"]
        country_folder = os.path.join(output_folder, country)
        os.makedirs(country_folder, exist_ok=True)  # Ensure country folder exists
        new_file_path = os.path.join(country_folder, f"{country}_artists_{batch_number}.json")
        
        # Open the new file and write the beginning of the array
        country_info["file_handle"] = open(new_file_path, 'w', encoding='utf-8')
        country_info["file_handle"].write('[')
        
        # Reset artist count and increment batch number for the next file
        country_info["artist_count"] = 0
        country_info["batch_number"] += 1
        country_info["first_item"] = True  # Reset the first_item flag for the new file
    
    return country_info["file_handle"]

def close_all_files():
    """Close all open file handles and properly end JSON arrays."""
    for country, info in file_info.items():
        if info["file_handle"]:
            info["file_handle"].write(']')  # Close the JSON array
            info["file_handle"].close()

def fill_missing_fields(artist, keys_to_keep):
    """Fill missing fields with empty strings."""
    filtered_artist = {}
    for key in keys_to_keep:
        # Fill with an empty string if the key is missing
        filtered_artist[key] = artist.get(key, "")
    
    if "genres" in artist and len(artist["genres"]) > 0:
        filtered_artist["genres"] = artist["genres"]
    elif "dbp_genre" in artist and len(artist["dbp_genre"]) > 0:
        filtered_artist["genres"] = artist["dbp_genre"]
    else:
        filtered_artist["genres"] = "[]"
    
    filtered_artist["country"] = country
    return filtered_artist

with open('data/artist-without-members.json', 'r', encoding='utf-8') as input_file:
    # Parse the JSON array incrementally
    parser = ijson.items(input_file, 'item')

    artist_done = 0

    try:
        while True:  # We'll increment the counter inside the loop
            artist = next(parser)  # Get the next artist
            
            # Extract the relevant country
            country = get_country_from_artist(artist)

            # Filter the artist data based on the keys to keep, handling missing fields
            filtered_artist = fill_missing_fields(artist, keys_to_keep)

            # Get the output file for the country
            output_file = get_output_file_for_country(country)

            # Write to the appropriate file for the country
            if not file_info[country]["first_item"]:
                output_file.write(',')  # Add a comma before each new item after the first one
            
            # Write the filtered artist as a JSON object
            json.dump(filtered_artist, output_file, indent=4)

            # Update the flag that the first item has been written
            file_info[country]["first_item"] = False

            # Increment the artist counter
            artist_done += 1
            file_info[country]["artist_count"] += 1  # Increment the artist count for the current file

            # Print status every 1,000 artists
            if artist_done % 1000 == 0:
                print(f"{artist_done} artists processed.")

    except StopIteration:
        # When we reach the end of the JSON file, the iteration stops
        pass

# Close all open files after processing
close_all_files()

print(f"Filtered JSON files written to '{output_folder}'. Total artists processed: {artist_done}")


1000 artists processed.
2000 artists processed.
3000 artists processed.
4000 artists processed.
5000 artists processed.
6000 artists processed.
7000 artists processed.
8000 artists processed.
9000 artists processed.
10000 artists processed.
11000 artists processed.
12000 artists processed.
13000 artists processed.
14000 artists processed.
15000 artists processed.
16000 artists processed.
17000 artists processed.
18000 artists processed.
19000 artists processed.
20000 artists processed.
21000 artists processed.
22000 artists processed.
23000 artists processed.
24000 artists processed.
25000 artists processed.
26000 artists processed.
27000 artists processed.
28000 artists processed.
29000 artists processed.
30000 artists processed.
31000 artists processed.
32000 artists processed.
33000 artists processed.
34000 artists processed.
35000 artists processed.
36000 artists processed.
37000 artists processed.
38000 artists processed.
39000 artists processed.
40000 artists processed.
41000 art

## Here is the script to filter albums data and group them by year then countries

In [5]:
import ijson
import json
import os
from collections import defaultdict

# List of keys to keep for each album
keys_to_keep = ["_id", "name", "id_artist", "title", "length", "genre", "cover" , "deezerFans"]
album_done = 0
output_folder = "data/processed_albums"
batch_size = 100  # Max number of albums per file

# Ensure the output folder exists
os.makedirs(output_folder, exist_ok=True)

# Dictionary to keep track of file handles, album counts, batch numbers, and first item flag for each year and country
file_info = defaultdict(lambda: defaultdict(lambda: {"file_handle": None, "album_count": 0, "batch_number": 1, "first_item": True}))
countries_set = set()  # Set to track all unique country names

def get_year_from_album(album):
    """Extract the year from the song's publication date or album date, handle '????' as empty."""
    if 'publicationDate' in album and album['publicationDate'] and not '?' in album['publicationDate'][:4] and 1000 < int(album['publicationDate'][:4]) < 2021:
        return album['publicationDate'][:4]  # Extract year (YYYY)
    elif 'publicationDateAlbum' in album and album['publicationDateAlbum'] and not '?' in album['publicationDateAlbum'][:4] and 1000 < int(album['publicationDateAlbum'][:4]) < 2021:
        return album['publicationDateAlbum'][:4]  # Assume it's just a year
    return "Unknown"

def get_country_from_album(album):
    """Extract the country from the album's location."""
    if 'country' in album and album['country']:
        return album['country']  # Return the country from the location field
    return "Unknown"  # Default to 'Unknown' if no country is found

def get_output_file_for_year_country(year, country):
    """Get or create the file for a specific year and country, batching them to 1,000 albums max per file."""
    country_info = file_info[year][country]
    
    # Check if we need to start a new batch file
    if country_info["file_handle"] is None or country_info["album_count"] >= batch_size:
        if country_info["file_handle"]:
            # Close the previous file's JSON array
            country_info["file_handle"].write(']')
            country_info["file_handle"].close()

        # Prepare the new file
        batch_number = country_info["batch_number"]
        year_folder = os.path.join(output_folder, year)
        os.makedirs(year_folder, exist_ok=True)  # Ensure year folder exists
        country_folder = os.path.join(year_folder, country)
        os.makedirs(country_folder, exist_ok=True)  # Ensure country folder exists
        new_file_path = os.path.join(country_folder, f"{country}_albums_{batch_number}.json")
        
        # Open the new file and write the beginning of the array
        country_info["file_handle"] = open(new_file_path, 'w', encoding='utf-8')
        country_info["file_handle"].write('[')
        
        # Reset album count and increment batch number for the next file
        country_info["album_count"] = 0
        country_info["batch_number"] += 1
        country_info["first_item"] = True  # Reset first item flag
    
    return country_info["file_handle"]

def close_all_files():
    """Close all open file handles and properly end JSON arrays."""
    for year, countries in file_info.items():
        for country, info in countries.items():
            if info["file_handle"]:
                info["file_handle"].write(']')  # Close the JSON array
                info["file_handle"].close()

def fill_missing_fields(album, keys_to_keep, country, year, date):
    """Fill missing fields with empty strings and add the country."""
    filtered_album = {}
    for key in keys_to_keep:
        # Fill with an empty string if the key is missing
        filtered_album[key] = album.get(key, "")
    filtered_album["country"] = country  # Add the country to the filtered album
    filtered_album["date"] = date
    filtered_album["dateRelease"] = year
    return filtered_album

with open('data/album.json', 'r', encoding='utf-8') as input_file:
    # Parse the JSON array incrementally
    parser = ijson.items(input_file, 'item')

    album_done = 0

    try:
        while True:
            album = next(parser)  # Get the next album
            
            # Extract the relevant year and country
            date = album.get("dateRelease", "")
            year = get_year_from_album(album)
            country = get_country_from_album(album)
            countries_set.add(country)  # Add the country to the set (to keep unique countries)

            # Filter the album data based on the keys to keep, handling missing fields
            filtered_album = fill_missing_fields(album, keys_to_keep, country, year, date)

            # Get the output file for the year and country
            output_file = get_output_file_for_year_country(year, country)

            # Write to the appropriate file for the year and country
            if not file_info[year][country]["first_item"]:
                output_file.write(',')  # Add a comma before each new item after the first one
            else:
                file_info[year][country]["first_item"] = False  # Set this to False after the first item for the year and country

            # Write the filtered album as a JSON object
            json.dump(filtered_album, output_file, ensure_ascii=False)

            # Increment the album counter
            album_done += 1
            file_info[year][country]["album_count"] += 1  # Increment the album count for the current file

            # Print status every 1,000 albums
            if album_done % 1000 == 0:
                print(f"{album_done} albums processed.")

    except StopIteration:
        # When we reach the end of the JSON file, the iteration stops
        pass

# Close all open files after processing
close_all_files()

# Print all unique countries at the end
print(f"Countries encountered: {', '.join(sorted(countries_set))}")

print(f"Filtered JSON files written to '{output_folder}'. Total albums processed: {album_done}")


1000 albums processed.
2000 albums processed.
3000 albums processed.
4000 albums processed.
5000 albums processed.
6000 albums processed.
7000 albums processed.
8000 albums processed.
9000 albums processed.
10000 albums processed.
11000 albums processed.
12000 albums processed.
13000 albums processed.
14000 albums processed.
15000 albums processed.
16000 albums processed.
17000 albums processed.
18000 albums processed.
19000 albums processed.
20000 albums processed.
21000 albums processed.
22000 albums processed.
23000 albums processed.
24000 albums processed.
25000 albums processed.
26000 albums processed.
27000 albums processed.
28000 albums processed.
29000 albums processed.
30000 albums processed.
31000 albums processed.
32000 albums processed.
33000 albums processed.
34000 albums processed.
35000 albums processed.
36000 albums processed.
37000 albums processed.
38000 albums processed.
39000 albums processed.
40000 albums processed.
41000 albums processed.
42000 albums processed.
4

### Here is how to get album folders hierarchy as a json to access it in JavaScript

In [6]:
import os
import json

def generate_folder_structure(root_dir='data/processed_albums'):
    structure = {}

    # Traverse through year folders in the root directory
    for year in os.listdir(root_dir):
        year_path = os.path.join(root_dir, year)
        if os.path.isdir(year_path):
            structure[year] = {}

            # Traverse through country folders in each year directory
            for country in os.listdir(year_path):
                country_path = os.path.join(year_path, country)
                if os.path.isdir(country_path):
                    structure[year][country] = []

                    # List all JSON files in each country folder
                    for file in os.listdir(country_path):
                        if file.endswith('.json'):
                            structure[year][country].append(file)

    # Save structure as JSON file
    with open('data/albums_structure.json', 'w') as f:
        json.dump(structure, f, indent=4)

if __name__ == '__main__':
    generate_folder_structure()
    print("Folder structure saved to albums_structure.json")


Folder structure saved to albums_structure.json


### Here is how to get artist folders hierarchy as a json to access it in JavaScript

In [7]:
import os
import json

def generate_folder_structure_artists(root_dir='data/processed_artists'):
    structure = {}

    # Traverse through country folders in the root directory
    for country in os.listdir(root_dir):
        country_path = os.path.join(root_dir, country)
        if os.path.isdir(country_path):
            structure[country] = []

            # List all JSON files in each country folder
            for file in os.listdir(country_path):
                if file.endswith('.json'):
                    structure[country].append(file)

    # Save structure as JSON file
    with open('data/artists_structure.json', 'w') as f:
        json.dump(structure, f, indent=4)

if __name__ == '__main__':
    generate_folder_structure_artists()
    print("Folder structure saved to artists_structure.json")


Folder structure saved to artists_structure.json


### Here is how to get song folders hierarchy as a json to access it in JavaScript

In [8]:
import os
import json

def generate_folder_structure_songs(root_dir='data/processed_songs'):
    structure = {}

    # Traverse through country folders in the root directory
    for year in os.listdir(root_dir):
        year_path = os.path.join(root_dir, year)
        if os.path.isdir(year_path):
            structure[year] = []

            # List all JSON files in each country folder
            for file in os.listdir(year_path):
                if file.endswith('.json'):
                    structure[year].append(file)

    # Save structure as JSON file
    with open('data/songs_structure.json', 'w') as f:
        json.dump(structure, f, indent=4)

if __name__ == '__main__':
    generate_folder_structure_songs()
    print("Folder structure saved to songs_structure.json")


Folder structure saved to songs_structure.json
